# Notebook 04: Data Coverage Analysis + Integration

**Goal:** 
1. Check how many product ingredients exist in our ingredient database
2. If coverage is good, enrich products with ingredient intelligence
3. Create final dataset for recommendation models

**Why this matters:**
- Products have ~10,000 unique ingredients
- Our ingredient DB has ~2,530 ingredients with ratings/categories
- Need to validate we have enough coverage to build a good recommender

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

## 1. Load Datasets

In [ ]:
# Load products
products = pd.read_parquet('../data/cleaned/skingen_products_lean_clean.parquet')
print(f"Loaded {len(products)} products")

# Load ingredients
ingredients_db = pd.read_parquet('../data/cleaned/ingredients_cleaned.parquet')
print(f"Loaded {len(ingredients_db)} ingredients with intelligence data")

In [ ]:
# Quick look at data
print("\nProduct columns:")
print(products.columns.tolist())

print("\nIngredient columns:")
print(ingredients_db.columns.tolist())

## 2. Coverage Analysis

Check how many product ingredients exist in our ingredient database.

In [ ]:
# Extract all unique ingredients from products
all_product_ingredients = set()

for ing_list in products['ingredient_list']:
    for ingredient in ing_list:
        all_product_ingredients.add(ingredient)

print(f"Unique ingredients across all products: {len(all_product_ingredients)}")

In [ ]:
# Get ingredient names from our database (lowercase for matching)
ingredient_db_names = set()
for name in ingredients_db['ingredient_name']:
    ingredient_db_names.add(name.lower())

print(f"Ingredients in our database: {len(ingredient_db_names)}")

In [ ]:
# Check overlap (case-insensitive matching)
product_ingredients_lower = set()
for ing in all_product_ingredients:
    product_ingredients_lower.add(ing.lower())

matched_ingredients = ingredient_db_names & product_ingredients_lower
missing_ingredients = product_ingredients_lower - ingredient_db_names

print("\nCoverage Statistics:")
print(f"  Matched: {len(matched_ingredients)} ({len(matched_ingredients)/len(product_ingredients_lower)*100:.1f}%)")
print(f"  Missing: {len(missing_ingredients)} ({len(missing_ingredients)/len(product_ingredients_lower)*100:.1f}%)")

### Which ingredients are we missing?

Let's check if missing ingredients are common/important or rare fillers.

In [ ]:
# Count how often each ingredient appears in products
ingredient_frequency = Counter()

for ing_list in products['ingredient_list']:
    for ingredient in ing_list:
        ingredient_frequency[ingredient.lower()] += 1

print(f"Total ingredient occurrences counted: {sum(ingredient_frequency.values())}")

In [ ]:
# Top 20 MISSING ingredients
print("Top 20 missing ingredients (by frequency):")
print("="*70)

missing_sorted = []
for ingredient, count in ingredient_frequency.most_common():
    if ingredient not in ingredient_db_names:
        missing_sorted.append((ingredient, count))

for ingredient, count in missing_sorted[:20]:
    print(f"{ingredient:45s}: {count:5d} products")

In [ ]:
# Top 20 MATCHED ingredients
print("\nTop 20 matched ingredients (by frequency):")
print("="*70)

matched_sorted = []
for ingredient, count in ingredient_frequency.most_common():
    if ingredient in ingredient_db_names:
        matched_sorted.append((ingredient, count))

for ingredient, count in matched_sorted[:20]:
    print(f"{ingredient:45s}: {count:5d} products")

### Per-Product Coverage

How many ingredients per product can we actually analyze?

In [ ]:
# Calculate coverage for each product
coverage_stats = []

for idx, row in products.iterrows():
    ingredient_list = row['ingredient_list']
    total_ingredients = len(ingredient_list)
    
    if total_ingredients == 0:
        coverage = 0
    else:
        matched_count = 0
        for ingredient in ingredient_list:
            if ingredient.lower() in ingredient_db_names:
                matched_count += 1
        
        coverage = matched_count / total_ingredients
    
    coverage_stats.append({
        'product_name': row['name'],
        'total_ingredients': total_ingredients,
        'matched_ingredients': matched_count if total_ingredients > 0 else 0,
        'coverage_pct': coverage * 100
    })

coverage_df = pd.DataFrame(coverage_stats)

In [ ]:
# Summary statistics
print("Per-Product Coverage Summary:")
print("="*70)
print(f"Average coverage: {coverage_df['coverage_pct'].mean():.1f}%")
print(f"Median coverage: {coverage_df['coverage_pct'].median():.1f}%")
print(f"Min coverage: {coverage_df['coverage_pct'].min():.1f}%")
print(f"Max coverage: {coverage_df['coverage_pct'].max():.1f}%")

print("\nProducts by coverage level:")
print(f"  0% coverage (no matches): {(coverage_df['coverage_pct'] == 0).sum()}")
print(f"  1-25% coverage: {((coverage_df['coverage_pct'] > 0) & (coverage_df['coverage_pct'] <= 25)).sum()}")
print(f"  26-50% coverage: {((coverage_df['coverage_pct'] > 25) & (coverage_df['coverage_pct'] <= 50)).sum()}")
print(f"  51-75% coverage: {((coverage_df['coverage_pct'] > 50) & (coverage_df['coverage_pct'] <= 75)).sum()}")
print(f"  76-100% coverage: {(coverage_df['coverage_pct'] > 75).sum()}")

In [ ]:
# Visualize coverage distribution
plt.figure(figsize=(10, 6))
plt.hist(coverage_df['coverage_pct'], bins=20, edgecolor='black')
plt.xlabel('Coverage (%)')
plt.ylabel('Number of Products')
plt.title('Distribution of Ingredient Coverage per Product')
plt.axvline(coverage_df['coverage_pct'].mean(), color='red', linestyle='--', label=f'Mean: {coverage_df['coverage_pct'].mean():.1f}%')
plt.legend()
plt.tight_layout()
plt.show()

### Decision: Can We Proceed?

Based on coverage analysis above, decide if we have enough data to build a good recommender.

In [ ]:
# Decision criteria
avg_coverage = coverage_df['coverage_pct'].mean()
products_with_good_coverage = (coverage_df['coverage_pct'] >= 50).sum()

print("\nDECISION CRITERIA:")
print("="*70)
print(f"Average coverage: {avg_coverage:.1f}%")
print(f"Products with ≥50% coverage: {products_with_good_coverage} ({products_with_good_coverage/len(products)*100:.1f}%)")

if avg_coverage >= 40 and products_with_good_coverage >= 5000:
    print("\n✅ PROCEED: Coverage is sufficient for recommendation system")
    proceed = True
elif avg_coverage >= 25:
    print("\n⚠️  CAUTION: Coverage is moderate - recommendations may be partial")
    print("   Consider: Add fallback logic for unknown ingredients")
    proceed = True
else:
    print("\n❌ STOP: Coverage too low - need more ingredient data")
    print("   Recommendation: Scrape more ingredients from Paula's Choice")
    proceed = False

## 3. Data Integration

If coverage is acceptable, enrich products with ingredient intelligence.

In [ ]:
# Create ingredient lookup dictionary for fast access
ingredient_lookup = {}

for idx, row in ingredients_db.iterrows():
    key = row['ingredient_name'].lower()
    ingredient_lookup[key] = {
        'rating': row['rating'],
        'benefits': row['benefits'],
        'categories': row['categories'],
        'functional_group': row['functional_group']
    }

print(f"Created lookup dictionary with {len(ingredient_lookup)} ingredients")

### Helper Functions

In [ ]:
# Helper function: Get ingredient data
def get_ingredient_data(ingredient_name):
    """Look up ingredient in database"""
    key = ingredient_name.lower()
    return ingredient_lookup.get(key, None)

# Test it
test_data = get_ingredient_data('Niacinamide')
print("Test lookup (Niacinamide):")
print(test_data)

In [ ]:
# Helper function: Count functional groups
def count_functional_groups(ingredient_list):
    """Count how many ingredients belong to each functional group"""
    counts = {
        'Actives': 0,
        'Support': 0,
        'Utility': 0,
        'Sensory': 0,
        'Risks': 0
    }
    
    for ingredient in ingredient_list:
        ing_data = get_ingredient_data(ingredient)
        if ing_data:
            for group in ing_data['functional_group']:
                if group in counts:
                    counts[group] += 1
    
    return counts

# Test it
test_ingredients = ['Niacinamide', 'Hyaluronic Acid', 'Glycerin']
test_counts = count_functional_groups(test_ingredients)
print("\nTest functional group counts:")
print(test_counts)